In [1]:
pip install ultralytics timm opencv-python torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.6 MB/s eta 0:00:00


In [2]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nirmalsankalana/plantdoc-dataset
!unzip plantdoc-dataset.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/plantdoc-dataset
License(s): CC0-1.0
 97% 868M/896M [00:07<00:00, 45.5MB/s]
100% 896M/896M [00:07<00:00, 121MB/s] 
Archive:  plantdoc-dataset.zip
  inflating: file_renamer.py         
  inflating: folder_renamer.py       
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_1.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_10.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_2.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_3.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_4.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_5.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_6.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_7.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_8.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_9.jpg  
  inflating: test/Apple_leaf/test_Apple leaf_1.jpg  

In [6]:
import os
import cv2
import numpy as np
from ultralytics import SAM

INPUT_ROOT = "train"
OUTPUT_ROOT = "segmented_dataset"

selected_classes = [
    "Bell_pepper_leaf",
    "Bell_pepper_leaf_spot",
    "Potato_leaf_late_blight",
    "Potato_leaf_early_blight"
]

sam_model = SAM("sam2_l.pt")

def segment_leaf(img_path):
    image = cv2.imread(img_path)
    h, w = image.shape[:2]

    results = sam_model(img_path)
    masks = results[0].masks.data.cpu().numpy()

    if len(masks) == 0:
        return image

    masks = [m for m in masks if m.sum() > 0.02 * h * w]

    if len(masks) == 0:
        return image

    mask = max(masks, key=lambda x: x.sum())
    mask = (mask * 255).astype(np.uint8)

    segmented = cv2.bitwise_and(image, image, mask=mask)
    segmented[mask == 0] = [255, 255, 255]

    ys, xs = np.where(mask > 0)
    y1, y2 = ys.min(), ys.max()
    x1, x2 = xs.min(), xs.max()

    pad = 0.05
    pad_y = int((y2 - y1) * pad)
    pad_x = int((x2 - x1) * pad)

    y1 = max(0, y1 - pad_y)
    y2 = min(h, y2 + pad_y)
    x1 = max(0, x1 - pad_x)
    x2 = min(w, x2 + pad_x)

    cropped = segmented[y1:y2, x1:x2]

    return cropped

for cls in selected_classes:

    input_folder = os.path.join(INPUT_ROOT, cls)

    if not os.path.exists(input_folder):
        print(f"Skipping missing class: {cls}")
        continue

    output_folder = os.path.join(OUTPUT_ROOT, cls)
    os.makedirs(output_folder, exist_ok=True)

    print(f"Processing class: {cls}")

    for img_name in os.listdir(input_folder):
        img_path = os.path.join(input_folder, img_name)
        save_path = os.path.join(output_folder, img_name)

        try:
            segmented_img = segment_leaf(img_path)
            cv2.imwrite(save_path, segmented_img)
        except Exception as e:
            print(f"Error processing {img_name}: {e}")

print("Segmentation completed")


Processing class: Bell_pepper_leaf

image 1/1 /content/train/Bell_pepper_leaf/train_Bell_pepper leaf_36_1.jpg: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 1 28, 1 29, 1 30, 1 31, 1 32, 1 33, 1 34, 1 35, 1 36, 1 37, 1 38, 1 39, 1 40, 1 41, 1 42, 1 43, 1 44, 1 45, 1 46, 1 47, 1 48, 1 49, 1 50, 1 51, 1 52, 1 53, 1 54, 1 55, 1 56, 1 57, 17501.9ms
Speed: 11.7ms preprocess, 17501.9ms inference, 19.3ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /content/train/Bell_pepper_leaf/train_Bell_pepper leaf_3.jpg: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 16971.8ms
Speed: 5.3ms preprocess, 16971.8ms inference, 1.9ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /content/train/Bell_pepper_leaf/train_Bell_pepper leaf_29.jpg: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8,

In [7]:
import shutil
from google.colab import files

shutil.make_archive('segmented_dataset', 'zip', 'segmented_dataset')
files.download('segmented_dataset.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import os
import cv2
import numpy as np

DATASET_ROOT = "segmented_dataset"

def is_bad_image(img_path):
    img = cv2.imread(img_path)

    if img is None:
        return True

    white_pixels = np.sum(np.all(img > 240, axis=2))
    total_pixels = img.shape[0] * img.shape[1]

    white_ratio = white_pixels / total_pixels

    if white_ratio > 0.90:
        return True

    return False


removed_count = 0

for cls in os.listdir(DATASET_ROOT):
    class_path = os.path.join(DATASET_ROOT, cls)

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        if is_bad_image(img_path):
            os.remove(img_path)
            removed_count += 1

print(f"Removed {removed_count} bad images")

Removed 2 bad images


In [9]:
import os
import shutil
import random

INPUT_ROOT = "segmented_dataset"
OUTPUT_ROOT = "segmented_split"

VAL_RATIO = 0.2

for cls in os.listdir(INPUT_ROOT):

    class_path = os.path.join(INPUT_ROOT, cls)
    images = os.listdir(class_path)
    random.shuffle(images)

    split_idx = int(len(images) * (1 - VAL_RATIO))

    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    train_dir = os.path.join(OUTPUT_ROOT, "train", cls)
    val_dir = os.path.join(OUTPUT_ROOT, "val", cls)

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(train_dir, img))

    for img in val_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(val_dir, img))

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
IMG_SIZE = 224
EPOCHS = 20
DATA_DIR = "segmented_split"

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = timm.create_model("efficientnet_b3", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 4)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

best_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Val Acc: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_model.pth")

print("Training completed")
print("Best validation accuracy:", best_acc)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1/20 | Loss: 1.1881 | Val Acc: 62.77%
Epoch 2/20 | Loss: 0.8791 | Val Acc: 63.83%
Epoch 3/20 | Loss: 0.7144 | Val Acc: 64.89%
Epoch 4/20 | Loss: 0.5776 | Val Acc: 59.57%
Epoch 5/20 | Loss: 0.5268 | Val Acc: 58.51%
Epoch 6/20 | Loss: 0.5720 | Val Acc: 57.45%
Epoch 7/20 | Loss: 0.5413 | Val Acc: 60.64%
Epoch 8/20 | Loss: 0.4784 | Val Acc: 60.64%
Epoch 9/20 | Loss: 0.4648 | Val Acc: 68.09%
Epoch 10/20 | Loss: 0.4552 | Val Acc: 60.64%
Epoch 11/20 | Loss: 0.4726 | Val Acc: 58.51%
Epoch 12/20 | Loss: 0.5007 | Val Acc: 60.64%
Epoch 13/20 | Loss: 0.4637 | Val Acc: 59.57%
Epoch 14/20 | Loss: 0.4584 | Val Acc: 60.64%
Epoch 15/20 | Loss: 0.4762 | Val Acc: 55.32%
Epoch 16/20 | Loss: 0.4344 | Val Acc: 56.38%
Epoch 17/20 | Loss: 0.4667 | Val Acc: 62.77%
Epoch 18/20 | Loss: 0.4604 | Val Acc: 62.77%
Epoch 19/20 | Loss: 0.4388 | Val Acc: 62.77%
Epoch 20/20 | Loss: 0.4220 | Val Acc: 60.64%
Training completed
Best validation accuracy: 68.08510638297872


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.43      0.43      0.43         7
           1       0.60      0.80      0.69        15
           2       0.54      0.47      0.50        32
           3       0.69      0.68      0.68        40

    accuracy                           0.61        94
   macro avg       0.56      0.59      0.57        94
weighted avg       0.60      0.61      0.60        94

[[ 3  3  1  0]
 [ 1 12  1  1]
 [ 1  5 15 11]
 [ 2  0 11 27]]
